# Step 1 — Load the Model

`all-MiniLM-L6-v2` is a good default: fast, small, and produces 384-dimensional embeddings. You can experiment with larger models later (e.g., `all-mpnet-base-v2` for 768 dimensions). The dimensionality is simply the length of the vector (matrix complexity).

In [4]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLm-L6-v2') # This downloads ~80MB on first run and caches locally

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

# Step 2 — Create Text Descriptions Per Track

This step is a form of feature fusion: structured audio features and unstructured metadata are combined into a single text representation before embedding.

For each track, build a text string combining metadata:

Why include audio feature values in the text? Because when the user types "high energy" you want the model to associate that with tracks that have high energy values, even if those tracks have no word "energy" in their title. The combined description creates a bridge.

For a row like:

```python
track_name     Blinding Lights
artists        The Weeknd
track_genre    pop
valence        0.34
energy         0.73
danceability   0.51
tempo          171
mode           1
```

the function builds:

```python
parts = [
    "Blinding Lights",
    "by The Weeknd",
    "genre: pop",
    "valence: 0.34",
    "energy: 0.73",
    "danceability: 0.51",
    "tempo: 171 BPM",
    "major key"
]
```

In [ ]:
import pandas as pd

df = pd.read_csv('../data/processed/tracks_engineered.csv')

def build_track_description(row):
    parts = [
        row['track_name'],
        f"by {row['artists']}",
        f"genre: {row['track_genre']}",
        f"valence: {row['valence']:.2f}",
        f"energy: {row['energy']:.2f}",
        f"danceability: {row['danceability']:.2f}",
        f"tempo: {row['tempo']:.0f} BPM",
        "major key" if row['mode'] == 1 else "minor key"
    ]
    return ' '.join(parts)

df['text_description'] = df.apply(build_track_description, axis=1) # Iterate through each track (row) and generate a text description from its columns
print(df['text_description'].iloc[0])

Comedy by Gen Hoshino genre: acoustic valence: 0.71 energy: 0.46 danceability: 0.68 tempo: 88 BPM minor key


# Step 3 — Encode All Tracks (Batch Processing)

Encoding ~600k tracks may take significant time. Use batching and `tqdm` to illustrate progress bar.

In this example, we are batching at every 512 rows because batching improves throughput and keeps memory usage stable by processing multiple sequences in one forward pass, and GPUs are especially good at parallel computation (e.g. matrix multiplications), although there is an exquisite balance memory consideration (each model forward pass stores tokenized inputs, attention matrices, and intermediate activations or hidden states).

For context, memory use is roughly:
```python
batch_size × sequence_length² × hidden_size (due to attention)
```

For every batch, a CPU could take tens of minutes to hours, whereas GPUs would take a few minutes:

```python
Text
 ↓
Tokenizer
 ↓
MiniLM transformer
 ↓
384-dimensional embedding
```

SentenceTransformer internally uses PyTorch, but we are using `convert_to_numpy=True` in this example since NumPy is CPU-native (PyTorch runs on both GPU and CPU), and FAISS expects NumPy arrays which allows for faster interoperability with other Python libraries and simpler debugging. The rule of thumb is to use PyTorch for training/internal testing & inference & fine-tuning while obviously being on CUDA, and NumPy for storing embeddings / preprocessing / analysis. In this pipeline, embeddings are converted to NumPy because FAISS expects NumPy arrays and for simpler downstream processing.

Notice how we are only running a small model here (circumstantial to number of songs and attributes). For frontier AI training, computation is distributed across GPU clusters. Each device processes different batches of data, computes gradients, and synchronizes them at every training step using gradient averaging.

You can expect the average standardized LLMs to be trained with the following pipeline:
```python
Text
 ↓
Model
 ↓
Loss
 ↓
Gradients
 ↓
Updated model
```

> **Warning:** This step may take 20–60 minutes depending on your hardware. Start it, grab a coffee, and let it run. If you have a GPU, `sentence-transformers` will use it automatically.

In [5]:
from tqdm import tqdm
import numpy as np

descriptions = df['text_description'].tolist() # Generate the 'text_description' column into a list
batch_size = 512 # Process 512 descriptions at a time
all_embeddings = []

for i in tqdm(range(0, len(descriptions), batch_size)): # From 0 to how many descriptions we have, update progress bar per each 512 completions
    batch = descriptions[i:i + batch_size] # First iteration: descriptions[0:512], second iteration: descriptions[512:1024]
    embeddings = model.encode( # Using SentenceTransformer('all-MiniLm-L6-v2') to encode()
        batch,
        convert_to_numpy=True, # False gets you a PyTorch tensors, True to NumPy arrays
        show_progress_bar=False) # Already using tqdm for progress so setting this to False (SentenceTransformer’s progress bar shows internal model processing, while tqdm tracks external batching.)
    all_embeddings.append(embeddings)

text_matrix = np.vstack(all_embeddings) # Gathers all batches into one big matrix
print(f"Text matrix shape: {text_matrix.shape}") # Should be (N, 384)

100%|██████████| 176/176 [05:16<00:00,  1.80s/it]

Text matrix shape: (89740, 384)


# Step 4 — L2-Normalize the Text Embeddings

For cosine similarity search, we L2-normalize vectors so that similarity depends on direction (meaning), not magnitude.

General rule of thumb:

Embeddings (text, audio, multimodal dense vectors)
    → use L2 normalization
    → enables cosine similarity via dot product

Sparse feature vectors (counts, TF-IDF, bag-of-words)
    → sometimes L1 normalization
    → preserves proportional distributions

Clipping / bounded feature scaling
    → max norm (rare in retrieval systems)

In [ ]:
from sklearn.preprocessing import normalize

text_matrix_normalized = normalize(text_matrix, norm='l2') # Forcing every vector (each row in text_matrix) to have unit length under the L2 norm where x_normalized = x / sqrt(a² + b² + c²)
np.save('../embeddings/text_matrix.npy', text_matrix_normalized)
print("Text matrix saved.")

Text matrix saved.


# Step 5 — Build the Hybrid Matrix

Combine audio features and text embeddings into a single vector per track.

Each track becomes a single point in a shared vector space:
- first part = semantic meaning (text embedding)
- second part = acoustic properties (audio features)

We scale each part before merging:
- `ALPHA` controls the influence of semantic similarity
- `BETA` controls the influence of audio similarity

This does not change how queries work directly. Instead, it defines the geometry of the space where FAISS performs nearest-neighbor search.
- Higher `ALPHA` → clustering is driven more by meaning (mood/genre language)
- Higher `BETA`  → clustering is driven more by sound (energy, tempo, etc.)

In [ ]:
audio_matrix = np.load('../embeddings/audio_matrix.npy')

# Normalize audio matrix to unit length as well (so scales match)
audio_matrix_normalized = normalize(audio_matrix, norm='l2')

# Weight: 60% text embedding, 40% audio features
# Adjust these weights later during evaluation
ALPHA = 0.6  # text weight
BETA = 0.4   # audio weight

hybrid_matrix = np.hstack([
    ALPHA * text_matrix_normalized,
    BETA * audio_matrix_normalized
])

hybrid_matrix = normalize(hybrid_matrix, norm='l2') # Normalize the full hybrid too

np.save('../embeddings/hybrid_matrix.npy', hybrid_matrix)
print(f"Hybrid matrix shape: {hybrid_matrix.shape}") # Should be (N, 394)